# Layer 8: 3CosAdd vs. 3CosMul

**Question:** does the choice of analogy scoring rule change the past-tense results for wav2vec2-base, layer 8?

- **3CosAdd** (Mikolov et al. 2013) is the paper's rule. It ranks every word token by its cosine to *b + a\* − a*.
- **3CosMul** (Levy & Goldberg 2014) ranks by cos(x, a\*) · cos(x, b) / (cos(x, a) + ε), with each cosine shifted to [0, 1]. It is known to be less dominated by one large similarity term.

This notebook runs the same analysis as `notebooks/layers/past_tense_layers.ipynb` (the feasibility notebook's Part 4), but for **layer 8 only** and **raw wav2vec2 only** (the "Wav2Vec" model, no word probe), once per scoring rule. It then compares the two.

**Same trials.** Both runs draw their analogies with the same seed, so trial *i* of an experiment is the same *a : a\* :: b : ?* question in both files. That allows a direct, trial-by-trial comparison (section 5), not only a comparison of averages.

**What you need**
- 3CosAdd: `outputs/analogy/runs_id/librispeech-train-clean-100/w2v2_8/experiment_results.csv` (the full Alliance run).
- 3CosMul: `outputs/analogy/runs_id_3cosmul/librispeech-train-clean-100/w2v2_8/experiment_results.csv` (the Mac run, `scripts/mac/run_layers_3cosmul.py`). It holds only the four experiments the comparison uses: `basic`, `regular`, `random_to_NNS`, `random_to_VBZ`.
- Python with `pandas`, `numpy`, `matplotlib` (the `m2v-layers` env). No GPU or PyTorch.

**What it produces:** tables and figures in `outputs/feasibility/layer8_3cosadd_vs_3cosmul/`.

## 1. Setup

Imports the per-layer analysis (`scripts/layers/compare_layers.py`, as `L`) and the feasibility helpers (`C`). It keeps only the raw wav2vec2 model.

- `LAYER`: the layer to compare (8).
- `N_BOOT`: bootstrap resamples for the confidence intervals. 2000 matches the other notebooks; use e.g. 300 for a quick look.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# find the repo root (the folder that contains scripts/layers/)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/layers/compare_layers.py").exists())
sys.path.insert(0, str(REPO / "scripts/layers"))
import compare_layers as L      # per-layer analysis (this project)
C = L.C                         # feasibility common.py: paths, constants, rank_summary(), plotting helpers

LAYER = 8
N_BOOT = 2000
METHODS = {"3CosAdd": "3cosadd", "3CosMul": "3cosmul"}   # label -> analogy_method in the file paths
# Okabe-Ito blue / vermillion; checked for colour-blind separation (dataviz validate_palette.js)
COLORS = {"3CosAdd": "#0072B2", "3CosMul": "#D55E00"}
MARKERS = {"3CosAdd": "o", "3CosMul": "s"}

C.MODELS[:] = ["Wav2Vec"]       # raw wav2vec2 only (3CosMul was not run for the word probe)
OUT = C.FEAS / "layer8_3cosadd_vs_3cosmul"
TABLES, FIGURES = OUT / "tables", OUT / "figures"
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

def results_path(method):
    L.ANALOGY_METHOD = method
    return L.results_paths(LAYER)["Wav2Vec"]

def savefig(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURES / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(FIGURES / f"{name}.png")))

C.set_style()
pd.set_option("display.width", 160, "display.max_columns", 30)
print("repo:", REPO)

## 2. Check the results files

Both files must exist. For the 3CosMul run, `EXPERIMENTS.txt` says which experiments it scored (the Mac run does the four the comparison needs).

In [ ]:
for label, method in METHODS.items():
    p = results_path(method)
    print(f"{label}: {p.relative_to(REPO)}  exists={p.exists()}")
    assert p.exists(), f"missing {p}"
    exp = p.parent / "EXPERIMENTS.txt"
    if exp.exists():
        print("   experiments:", exp.read_text().strip().replace("\n", ", "))

## 3. Pair the trials and check the 3CosMul run is sane

This reads both files and lines up trial *i* of each experiment. The words must be identical in every pair; otherwise the two runs did not use the same trials, and section 5 would be meaningless.

It also checks that 3CosMul does **much better than chance**. If its median rank on `basic` is in the thousands, the 3CosMul run scored misaligned vectors (e.g. the first Mac run, extracted in the wrong utterance order; see `scripts/mac/recover_cluster_order.py`). The notebook then stops, because none of the comparisons below would mean anything.

In [ ]:
CHECK_SANITY = True   # set False only to test the notebook's code on a run you know is broken

raw = {}
for label, method in METHODS.items():
    C.RESULTS = {"Wav2Vec": results_path(method)}
    raw[label] = L.E.export_results("Wav2Vec")

experiments = sorted(set(raw["3CosMul"].experiment))
raw["3CosAdd"] = raw["3CosAdd"][raw["3CosAdd"].experiment.isin(experiments)]
keys = ["experiment", "trial_idx"]
words = ["base_from", "inflected_from", "base_to", "inflected_to"]
paired = raw["3CosAdd"][keys + words + ["group0", "group1", "gt_label_rank"]].merge(
    raw["3CosMul"][keys + words + ["gt_label_rank"]], on=keys, suffixes=("_add", "_mul"), how="inner")
same_words = np.all([paired[f"{w}_add"] == paired[f"{w}_mul"] for w in words], axis=0)
print(f"{len(paired):,} paired trials in {experiments}; same words in {same_words.mean():.1%} of them")
print({label: len(raw[label]) for label in METHODS}, "trials per file")
assert same_words.all() and len(paired) == len(raw["3CosMul"]) == len(raw["3CosAdd"]), "the two runs did not score the same trials"
paired = paired.rename(columns={f"{w}_add": w for w in words}).drop(columns=[f"{w}_mul" for w in words])

sanity = pd.DataFrame({label: raw[label].groupby("experiment").gt_label_rank.agg(
    top1=lambda r: (r == 0).mean(), median_rank="median") .stack() for label in METHODS}).unstack()
display(sanity.round(3))
if CHECK_SANITY:
    med = raw["3CosMul"].query("experiment == 'basic'").gt_label_rank.median()
    assert med < 100, (f"3CosMul median rank on 'basic' is {med:.0f}: chance level. The 3CosMul run scored "
                       "misaligned vectors; redo it before reading this notebook (see the markdown above).")

## 4. Conditions: past tense vs. -s, with each scoring rule

The same conditions as Figure 7 of the feasibility notebook and section 7 of the layers notebook, all *within-class* analogies:
- **past, regular**: regular [t]/[d]/[ɪd] pasts (automatic class), e.g. *walk : walked :: play : ?*
- **past, irregular**: irregular pasts without modals, *be* and homographs.
- **plural -s**, **3sg -s**: the paper's inflections.
- **random pair**: the control, an analogy whose difference vector is meaningless.

Measures: **top-1 accuracy** (share of trials where the correct word is the nearest; higher is better) and **mean rank** (lower is better; driven by a few very bad trials, so on a log scale). The 95% confidence intervals resample *verbs*, not trials.

In [ ]:
pairs = L.load_pairs()
tables = {}
for label, method in METHODS.items():
    print(label, "...", flush=True)
    L.ANALOGY_METHOD = method
    g, c, v = L.layer_tables(LAYER, pairs, N_BOOT)
    tables[label] = {"grid": g.assign(method=label), "cond": c.assign(method=label), "per_verb": v.assign(method=label)}
grid, cond, per_verb = (pd.concat([tables[m][k] for m in METHODS], ignore_index=True) for k in ("grid", "cond", "per_verb"))

SHOW = [L.COND_PAST_REG_CLEAN, L.COND_PAST_IRR_CLEAN, L.COND_NNS, L.COND_VBZ, L.COND_RANDOM]
NAMES = {k: v[3] for k, v in L.COND_STYLE.items()}
c = cond[cond.condition.isin(SHOW)].copy()
c["top-1 [95% CI]"] = c.apply(lambda r: f"{r.top1:.2f} [{r.top1_lo:.2f}, {r.top1_hi:.2f}]", axis=1)
c["mean rank [95% CI]"] = c.apply(lambda r: f"{r.mean_rank:.0f} [{r.mean_lo:.0f}, {r.mean_hi:.0f}]", axis=1)
cond_table = (c.assign(condition=c.condition.map(NAMES))
              .pivot_table(index="condition", columns="method", values=["top-1 [95% CI]", "mean rank [95% CI]", "median_rank", "n_trials"], aggfunc="first")
              .reindex([NAMES[k] for k in SHOW]))
display(cond_table)

In [ ]:
# Figure: each condition, both rules side by side, with 95% CIs
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
y = np.arange(len(SHOW))
for ax, (est, lo, hi, xlabel, log) in zip(axs, [("top1", "top1_lo", "top1_hi", "top-1 accuracy (higher is better)", False),
                                                ("mean_rank", "mean_lo", "mean_hi", "mean rank (log; lower is better)", True)]):
    for k, label in enumerate(METHODS):
        d = cond[cond.method == label].set_index("condition").reindex(SHOW)
        yy = y + (k - 0.5) * 0.28
        ax.errorbar(d[est], yy, xerr=[d[est] - d[lo], d[hi] - d[est]], fmt=MARKERS[label], ms=8, lw=2, capsize=0,
                    color=COLORS[label], label=label)
    ax.set_yticks(y, [NAMES[k] for k in SHOW]); ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    if log: ax.set_xscale("log")
    else: ax.set_xlim(0, 1)
    ax.grid(axis="y", visible=False)
axs[1].set_yticklabels([])
axs[0].legend(loc="lower right", frameon=False)
fig.suptitle(f"wav2vec2-base layer {LAYER}, raw hidden states: 3CosAdd vs. 3CosMul (bars = 95% item-bootstrap CI)")
fig.tight_layout()
savefig(fig, "fig_conditions_3cosadd_vs_3cosmul")

## 5. Trial by trial: when does 3CosMul help, and when does it hurt?

Because both rules scored the same trials, each trial can be compared directly: did the correct word move **up** (better rank with 3CosMul), **down**, or stay the same? This is more sensitive than comparing averages, because the trial-to-trial variation (some analogies are just hard) cancels out.

Rows are the experiment and the inflection being tested (`VBD` = past tense, `NNS` = plural, `VBZ` = 3sg). For `regular`, the second part says whether the source pair is regular (`True`) or irregular (`False`).
- **3CosMul better / worse / same**: share of trials.
- **top-1 gained / lost**: trials that became / stopped being exactly right.
- **median rank change**: 3CosMul rank − 3CosAdd rank; negative means 3CosMul is better.

In [ ]:
p = paired.assign(change=paired.gt_label_rank_mul - paired.gt_label_rank_add,
                  top1_add=paired.gt_label_rank_add == 0, top1_mul=paired.gt_label_rank_mul == 0,
                  inflection=paired.group0.fillna("–"),
                  regular=paired.group1.map(lambda g: "" if g is None or g != g else str(g)))
trial_cmp = p.groupby(["experiment", "inflection", "regular"]).agg(
    trials=("change", "size"),
    **{"3CosMul better": ("change", lambda x: (x < 0).mean()),
       "3CosMul worse": ("change", lambda x: (x > 0).mean()),
       "same": ("change", lambda x: (x == 0).mean()),
       "top-1 gained": ("top1_mul", lambda x: (x & ~p.loc[x.index, "top1_add"]).sum()),
       "top-1 lost": ("top1_add", lambda x: (x & ~p.loc[x.index, "top1_mul"]).sum()),
       "median rank change": ("change", "median")})
display(trial_cmp.round(3))

In [ ]:
# Figure: 3CosAdd rank vs. 3CosMul rank for every past-tense trial of the `regular` experiment.
# Points below the diagonal: 3CosMul ranked the correct word higher. symlog so rank 0 is visible.
d = p[(p.experiment == "regular") & (p.inflection == "VBD")]
fig, axs = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (reg, title) in zip(axs, [("True", "regular past → regular past"), ("False", "irregular past → irregular past")]):
    s = d[d.regular == reg]
    ax.scatter(s.gt_label_rank_add, s.gt_label_rank_mul, s=8, alpha=0.25, color="#555555", lw=0)
    m = max(d.gt_label_rank_add.max(), d.gt_label_rank_mul.max())
    ax.plot([0, m], [0, m], "--", color=C.CONTROL_COLOR, lw=1)
    ax.set_xscale("symlog", linthresh=1); ax.set_yscale("symlog", linthresh=1)
    better, worse = (s.gt_label_rank_mul < s.gt_label_rank_add).mean(), (s.gt_label_rank_mul > s.gt_label_rank_add).mean()
    ax.set_title(f"{title}\n{len(s):,} trials: 3CosMul better {better:.0%}, worse {worse:.0%}")
    ax.set_xlabel("rank with 3CosAdd (symlog)")
axs[0].set_ylabel("rank with 3CosMul (symlog)")
fig.suptitle(f"Layer {LAYER}, past tense: the same trials scored both ways (below the dashed line = 3CosMul better)")
fig.tight_layout()
savefig(fig, "fig_trials_regular_past")

## 6. [t] / [d] / [ɪd] and irregular pasts

Feasibility Figure 6, for each rule: rows are the source ending (*a : a\**), columns the target ending (*b : ?*), each cell its top-1 accuracy. The last row and column are irregular pasts (as in the layers notebook, section 9).

The third panel is the **difference** (3CosMul − 3CosAdd): orange means 3CosMul is better in that cell, purple means worse, white means no change. Cells with few trials (see `n_trials` in the saved table) are noisy.

In [ ]:
from matplotlib.colors import Normalize, TwoSlopeNorm
cats = L.CATEGORIES
def cell_matrix(label, est="top1"):
    s = grid[grid.method == label].set_index(["src", "tgt"])[est]
    return s.unstack().reindex(index=cats, columns=cats)

fmt = lambda x: f"{x:.2f}" if np.isfinite(x) else "–"
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, label in zip(axs, METHODS):
    v = cell_matrix(label)
    im = C.heatmap(ax, v, v.map(fmt), Normalize(0, 1), cmap="Blues", title=f"{label}: top-1 accuracy",
                   ylabel="source" if label == "3CosAdd" else "")
fig.colorbar(im, ax=axs[:2], shrink=0.8, label="top-1 accuracy")
diff = cell_matrix("3CosMul") - cell_matrix("3CosAdd")
lim = 1.4 * max(0.05, np.nanmax(np.abs(diff.to_numpy())))   # headroom so cell text stays readable
im2 = C.heatmap(axs[2], diff, diff.map(lambda x: f"{x:+.2f}" if np.isfinite(x) else "–"),
                TwoSlopeNorm(0, -lim, lim), cmap="PuOr_r", title="3CosMul − 3CosAdd", ylabel="")
fig.colorbar(im2, ax=axs[2], shrink=0.8, label="difference in top-1")
fig.suptitle(f"Layer {LAYER}, past tense: transfer across endings (irreg. = irregular pasts)")
savefig(fig, "fig_allomorph_grid_3cosadd_vs_3cosmul")

grid_table = grid.pivot_table(index=["src", "tgt"], columns="method", values=["top1", "mean_rank", "n_trials"]).reindex(
    pd.MultiIndex.from_product([cats, cats], names=["src", "tgt"])).dropna(how="all")
display(grid_table.round(3))

## 7. Which verbs change the most

The mean rank of every regular target verb under each rule, and the verbs whose mean rank changed the most. The layer-8 feasibility analysis found that a few verbs dominate the regular-past mean rank, e.g. ones whose past sounds like another word (*rowed* ~ *road*). Here we see whether 3CosMul fixes or worsens those.

In [ ]:
v = per_verb.groupby(["method", "base_to", "inflected_to", "past_sounds_like"], as_index=False).agg(
    mean_rank=("mean_rank", "mean"), trials=("trials", "sum"))
verbs = v.pivot_table(index=["base_to", "inflected_to", "past_sounds_like"], columns="method", values="mean_rank").reset_index()
verbs["change"] = verbs["3CosMul"] - verbs["3CosAdd"]
display(Markdown("**15 verbs most helped by 3CosMul** (most negative change)"))
display(verbs.sort_values("change").head(15).round(1))
display(Markdown("**15 verbs most hurt by 3CosMul**"))
display(verbs.sort_values("change", ascending=False).head(15).round(1))
display(Markdown("**The 15 worst verbs under 3CosAdd, and what 3CosMul does to them**"))
display(verbs.sort_values("3CosAdd", ascending=False).head(15).round(1))

## 8. Save the tables

Writes to `outputs/feasibility/layer8_3cosadd_vs_3cosmul/tables/` (small; meant to be committed, next to the figures):
- `conditions.csv`: section 4, both rules (long format).
- `trial_comparison.csv`: section 5.
- `allomorph_grid.csv`: section 6, both rules.
- `per_verb.csv`: section 7.

In [ ]:
cond.to_csv(TABLES / "conditions.csv", index=False)
trial_cmp.reset_index().to_csv(TABLES / "trial_comparison.csv", index=False)
grid.to_csv(TABLES / "allomorph_grid.csv", index=False)
verbs.to_csv(TABLES / "per_verb.csv", index=False)
for f in sorted(OUT.rglob("*.*")):
    print(f"{f.relative_to(REPO)}  ({f.stat().st_size / 1024:.0f} KB)")